# 05 - Explaining one model: MASTER, DJI, test year 2020

Step 5 of `THESIS_GUIDE.md` (PIANO.md B7). Before running the explanations on the whole grid, every method is looked at on one run: MASTER, DJI, T = 20, L = 5, seed 42, test year 2020.

**What is explained.** On every other weekly rebalance date of the test year (about 26 days, the days the top-k portfolio trades on), each stock's output is attributed to that stock's own inputs, a T x F window (20 days x 157 Alpha158 features + 63 market features). Two outputs (decision D4): the score (`pred`) and the **top-k margin** (`topk_margin`: the score minus the score the stock has to beat to be among the k = 5 held stocks), which explains portfolio membership rather than the score.

**Methods** (`xaifin/xai/`):
- **Integrated Gradients** (`gradient.py`): the path from a baseline to the input, 16 Gauss-Legendre steps. Baseline: the day's average stock (cross-sectional mean input), so an attribution says why this stock scores differently from the average stock of that day. Path "joint": the whole day moves from the baseline together and each stock keeps the gradient through its own inputs (decisions of 2026-11-04 in PIANO.md; section 6 checks them).
- **Saliency**: the gradient at the input.
- **Permutation importance by family** (`permutation.py`): the inputs of one family are shuffled across the stocks of the day; the drop in RankIC and the share of the top-k replaced measure what the family is worth for ranking and for the portfolio.
- **MASTER's own weights** (`intrinsic.py`): the feature gate, the temporal and the inter-stock attention, as supporting evidence only.

Importance is aggregated (`aggregate.py`) as the share of |attribution| each feature, lag or family gets, per stock-day, averaged.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin.analysis.agreement import agreement, agreement_matrix
from xaifin.config import RESULTS_ROOT, TOP_K
from xaifin.models.registry import load_run
from xaifin.xai.aggregate import global_importance, input_table
from xaifin.xai.base import explain_dates, explain_run, load_attributions, save_attributions, xai_dir
from xaifin.xai.gradient import IntegratedGradients, Saliency, completeness_gap
from xaifin.xai.intrinsic import run_weights
from xaifin.xai.permutation import FamilyPermutation, summarize

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
RUN = RESULTS_ROOT / "Regression/MASTER/dji/sl20_pl5/seed42/y2020"
adapter = load_run(RUN)
K = TOP_K[adapter.cfg.universe]
dates = explain_dates(adapter)
inputs = input_table(adapter)
names = np.array(adapter.feature_names)
print(f"{adapter.cfg.model} {adapter.cfg.universe} T={adapter.cfg.seq_len} L={adapter.cfg.pred_len} seed {adapter.cfg.seed} "
      f"y{adapter.cfg.test_year}: {len(dates)} explained days, k = {K}, {len(inputs)} inputs per stock")

## 1. Attributions

Read from the results store (`scripts/explain.py` writes them); computed and saved here if the run is not explained yet.

In [ ]:
methods = {"ig_joint": IntegratedGradients(), "saliency": Saliency()}
attr, glob = {}, {}
for name, explainer in methods.items():
    for target in ("pred", "topk_margin"):
        result = load_attributions(adapter, name, target)
        if result is None:
            result = explain_run(adapter, explainer, target, K, dates)
            out = save_attributions(adapter, name, target, result, explainer.params())
            global_importance(result["attr"], result["date"], inputs).to_parquet(out / f"global_{target}.parquet")
        attr[name, target] = result
        glob[name, target] = global_importance(result["attr"], result["date"], inputs)
print({key: r["attr"].shape for key, r in attr.items()})

## 2. One stock on one day

The IG attribution of the stock with the highest score on the first explained day, for the 30 inputs (feature x lag) with the largest attributions. Red: the input raises the score above the average stock's, blue: lowers it.

In [ ]:
r = attr["ig_joint", "pred"]
day = r["date"][0]
rows = np.flatnonzero(r["date"] == day)
i = rows[np.argmax(r["score"][rows])]
a = r["attr"][i]
top = np.argsort(-np.abs(a).sum(0))[:30]
lags = [f"lag {a.shape[0] - 1 - t}" for t in range(a.shape[0])]
px.imshow(a[:, top], x=names[top], y=lags, color_continuous_scale="RdBu_r", color_continuous_midpoint=0, aspect="auto",
          title=f"IG, {r['ticker'][i]} on {pd.Timestamp(day).date()} (highest score of the day)").show()

## 3. Which inputs matter: features, families, lags; score against top-k membership

Share of the |IG attribution| per feature (summed over lags), per family and per lag, averaged over stock-days; error bars: standard deviation over days. `signed`: the same share with its sign.

In [ ]:
g = {t: glob["ig_joint", t] for t in ("pred", "topk_margin")}
display(pd.concat({t: g[t][g[t]["level"] == "feature"].head(20)[["name", "importance", "signed"]].reset_index(drop=True) for t in g}, axis=1))
fam = pd.concat([g[t][g[t]["level"] == "family"].assign(target=t) for t in g])
px.bar(fam, x="name", y="importance", color="target", barmode="group", error_y="std_days",
       title="IG: share of the attribution per family").show()
lag = pd.concat([g[t][g[t]["level"] == "lag"].assign(target=t) for t in g])
lag["lag"] = lag["name"].astype(int)
px.line(lag.sort_values("lag"), x="lag", y="importance", color="target", markers=True,
        title="IG: share of the attribution per lag (0 = last input day)").show()
vec = lambda gl, level: gl[gl["level"] == level].set_index("name")["importance"]
pd.DataFrame({level: agreement(vec(g["pred"], level), vec(g["topk_margin"], level)) for level in ("feature", "family", "lag")}).T

## 4. Saliency and permutation importance against IG

Family rankings of the three methods (the market family is left out: the day-mean baseline and the cross-sectional shuffle cannot see it, see section 5). Permutation importance: `d_RankIC`, the drop in RankIC when the family is shuffled across stocks, and `topk_change`, the share of the 5 held stocks that change.

In [ ]:
perm_dir = xai_dir("permutation", adapter)
perm_rows = (pd.read_parquet(perm_dir / "permutation.parquet") if (perm_dir / "permutation.parquet").exists()
             else FamilyPermutation().explain_run(adapter, dates, K))
perm = summarize(perm_rows)
display(perm)
family = lambda gl: gl[(gl["level"] == "family") & (gl["name"] != "market")].set_index("name")["importance"]
vectors = {"IG": family(glob["ig_joint", "pred"]), "Saliency": family(glob["saliency", "pred"]),
           "IG top-k": family(glob["ig_joint", "topk_margin"]),
           "perm. top-k change": perm.set_index("family")["topk_change"], "perm. dRankIC": perm.set_index("family")["d_RankIC"]}
display(agreement_matrix(vectors, "spearman"))
feature = lambda gl: gl[gl["level"] == "feature"].set_index("name")["importance"]
agreement(feature(glob["ig_joint", "pred"]), feature(glob["saliency", "pred"]))

## 5. MASTER's own weights

The **feature gate** reweights the 157 Alpha158 features from the market features of the last input day (the same weights for every stock, mean 1): it is the channel through which the market features act, invisible to IG with the day-mean baseline. **Temporal attention**: the weight of each day of the window. **Inter-stock attention**: how much a stock attends to the others; 1/N for every pair would be uniform.

In [ ]:
if adapter.cfg.model == "MASTER":
    w = run_weights(adapter, dates)
    print("replay of the forward pass, largest difference from the model's output:", w["max_diff"])
    gate = pd.DataFrame(w["gate"], columns=names[:w["gate"].shape[1]], index=pd.to_datetime(w["date"]))
    gate_summary = gate.agg(["mean", "std"]).T.sort_values("mean", ascending=False)
    display(gate_summary.head(15))
    px.line(gate[gate_summary.index[:5]], title="Feature gate of the 5 most up-weighted features, per explained day").show()
    temporal = np.concatenate(w["temporal"]).mean(0)
    px.bar(x=[len(temporal) - 1 - t for t in range(len(temporal))], y=temporal, labels={"x": "lag", "y": "mean weight"},
           title="Temporal attention per lag (0 = last input day)").show()
    self_weight = [np.diag(m).mean() * len(m) for m in w["inter"]]
    print(f"inter-stock attention on oneself, relative to uniform (1 = uniform): {np.mean(self_weight):.2f}")
    ig_feat = feature(glob["ig_joint", "pred"])
    print("gate mean vs IG feature importance (Alpha158 features):", agreement(gate_summary["mean"], ig_feat))

## 6. Checks of the IG settings

On the first three explained days: (a) the path, 'joint' (used) against the guide's 'own' (only the stock's inputs move, the other stocks stay; exact completeness per stock); (b) the baseline, the day's average stock (used) against zero (the training median); (c) completeness of 'own' (sum of the attributions minus the change of the target from the baseline), which also checks that 16 steps suffice.

In [ ]:
checks = []
for batch in adapter.day_batches("test"):
    if batch.date not in dates[:3]:
        continue
    joint = IntegratedGradients(path="joint").explain(adapter, batch, "pred", K)
    own = IntegratedGradients(path="own").explain(adapter, batch, "pred", K)
    zero = IntegratedGradients(path="joint", baseline="zero").explain(adapter, batch, "pred", K)
    gap = completeness_gap(adapter, batch, own, "pred", K, baseline="mean")
    for i in range(len(batch.tickers)):
        s = lambda v: pd.Series(np.abs(v[i]).sum(0), index=names)
        checks.append({"date": batch.date, "own vs joint": agreement(s(own), s(joint))["spearman"],
                       "zero vs mean baseline": agreement(s(zero), s(joint))["spearman"],
                       "completeness gap / |change|": abs(gap[i]) / max(abs(own[i].sum() - gap[i]), 1e-12)})
pd.DataFrame(checks).describe().loc[["mean", "25%", "50%", "75%", "min"]]

## Observations

*To be written once the run (MASTER, DJI, seed 42, test year 2020) is retrained and explained (PIANO.md B7).*